<a href="https://colab.research.google.com/github/PARIJAAT-13/Flyrank-A.I/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Finding 1 — Freshness and Impressions

The paper reports that refreshed mature pages showed substantially higher impressions than stale pages.

Methodology question: How was “refreshed” defined, and were refreshed and stale pages comparable before the refresh? Could page age, client/site differences, or existing performance explain part of the observed difference? The validation should support this as an observed association rather than proof that refreshing caused the increase.

### Finding 2 — Search Volume and Traffic

The paper reports that high search volume was not a reliable page-level traffic forecast, with the share of pages exceeding their search-volume estimate decreasing as search-volume buckets increased.

Methodology question: How were search-volume estimates matched to the page-level traffic measurement window, and were differences between pages, clients, and page age accounted for? The validation should support this as a directional comparison rather than a causal claim.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
paper_available = True
warehouse_dataset_available = False

print("Research paper available:", paper_available)
print("Warehouse dataset available:", warehouse_dataset_available)

if not paper_available:
    print("⚠️ Paper methodology cannot be audited yet without inventing evidence.")

if not warehouse_dataset_available:
    print("⚠️ Warehouse dataset resource is still pending review.")

Research paper available: True
Warehouse dataset available: False
⚠️ Warehouse dataset resource is still pending review.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Split design

I use a grouped split by client_id so that content from the same client does not appear in both training and validation sets. This is more conservative than a random row split because the model is evaluated on clients it did not train on.

The Week-5 model and baseline are evaluated on the same validation rows and with the same MAE metric. The comparison is directional: the goal is to measure whether the learned model generalizes beyond the easier random-split setting, not to claim production performance.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/PARIJAAT-13/Flyrank-A.I.git"
REPO_ROOT = Path("/content/Flyrank-A.I")

if not REPO_ROOT.exists():
    print("Cloning Flyrank-A.I repository...")
    !git clone {REPO_URL} /content/Flyrank-A.I
else:
    print("Repository already exists.")

data_path = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"

print("\nRepository exists:", REPO_ROOT.exists())
print("Dataset exists:", data_path.exists())

if data_path.exists():
    print("Dataset path:", data_path)
else:
    print("\n❌ Dataset not found.")
    print("Checking data/raw:")
    data_dir = REPO_ROOT / "data/raw"
    if data_dir.exists():
        print(list(data_dir.iterdir()))

Repository already exists.

Repository exists: True
Dataset exists: True
Dataset path: /content/Flyrank-A.I/data/raw/content_refresh_anonymized.csv


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---------------------------------------------------------
# 1. Locate repository and dataset
# ---------------------------------------------------------

repo_root = Path("/content/Flyrank-A.I")

data_path = repo_root / "data/raw/content_refresh_anonymized.csv"

if not data_path.exists():
    # Fallback search in case the repository is mounted elsewhere
    matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))
    if not matches:
        raise FileNotFoundError(
            "content_refresh_anonymized.csv was not found under /content."
        )
    data_path = matches[0]

print("Dataset:", data_path)

df = pd.read_csv(data_path)

print(f"Dataset rows: {len(df):,}")
print(f"Dataset columns: {len(df.columns):,}")

# ---------------------------------------------------------
# 2. Prepare the same features used in Week 5
# ---------------------------------------------------------

features = [
    "days_since_last_update",
    "search_volume",
    "impressions_last_30d",
    "avg_position"
]

for col in features:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# ---------------------------------------------------------
# 3. Recreate the Week-4 baseline score
# ---------------------------------------------------------

df["staleness_score"] = df["days_since_last_update"].rank(pct=True)
df["volume_score"] = df["search_volume"].rank(pct=True)
df["impression_score"] = df["impressions_last_30d"].rank(pct=True)

df["position_score"] = (
    (df["avg_position"] >= 4) &
    (df["avg_position"] <= 20)
).astype(float)

df["baseline_score"] = (
    0.40 * df["staleness_score"]
    + 0.30 * df["volume_score"]
    + 0.20 * df["impression_score"]
    + 0.10 * df["position_score"]
)

# ---------------------------------------------------------
# 4. Honest grouped split by client
# ---------------------------------------------------------

X = df[features]
y = df["baseline_score"]
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("\n=== GROUPED SPLIT ===")
print(f"Training rows:   {len(X_train):,}")
print(f"Validation rows: {len(X_test):,}")
print(f"Training clients:   {len(train_clients):,}")
print(f"Validation clients: {len(test_clients):,}")
print(
    "Client overlap:",
    len(train_clients.intersection(test_clients))
)

# ---------------------------------------------------------
# 5. Train Week-5 Random Forest
# ---------------------------------------------------------

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_pred = model.predict(X_test)

# ---------------------------------------------------------
# 6. Compare model vs baseline on SAME validation rows
# ---------------------------------------------------------

baseline_pred = y_test.values

baseline_mae = mean_absolute_error(
    y_test,
    baseline_pred
)

model_mae = mean_absolute_error(
    y_test,
    model_pred
)

baseline_rmse = mean_squared_error(
    y_test,
    baseline_pred
) ** 0.5

model_rmse = mean_squared_error(
    y_test,
    model_pred
) ** 0.5

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Week-5 Random Forest"
    ],
    "MAE": [
        baseline_mae,
        model_mae
    ],
    "RMSE": [
        baseline_rmse,
        model_rmse
    ]
})

print("\n=== BEFORE / AFTER COMPARISON ===")
display(comparison)

# ---------------------------------------------------------
# 7. Directional interpretation
# ---------------------------------------------------------

mae_change = model_mae - baseline_mae

print("\n=== INTERPRETATION ===")

if mae_change < 0:
    print(
        f"The Random Forest reduced MAE by "
        f"{abs(mae_change):.6f} on the grouped validation set."
    )
elif mae_change > 0:
    print(
        f"The Random Forest increased MAE by "
        f"{mae_change:.6f} on the grouped validation set."
    )
else:
    print("The Random Forest and baseline have the same MAE.")

print(
    "\nThis is a grouped validation result and should be treated "
    "as directional evidence of generalization, not as a production guarantee."
)

Dataset: /content/Flyrank-A.I/data/raw/content_refresh_anonymized.csv
Dataset rows: 30,000
Dataset columns: 44

=== GROUPED SPLIT ===
Training rows:   23,837
Validation rows: 6,163
Training clients:   25
Validation clients: 7
Client overlap: 0

=== BEFORE / AFTER COMPARISON ===


,method,MAE,RMSE
0,Week-4 baseline,0.000000,0.000000
1,Week-5 Random Forest,0.006802,0.016359



=== INTERPRETATION ===
The Random Forest increased MAE by 0.006802 on the grouped validation set.

This is a grouped validation result and should be treated as directional evidence of generalization, not as a production guarantee.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
### Leakage audit

I checked the final feature set against the available columns and excluded fields that represent future outcomes or future performance windows.

The final model uses only:
- days_since_last_update
- search_volume
- impressions_last_30d
- avg_position

I also verify that no outcome-derived or future-window fields are included in the model features. The grouped split keeps clients separated between training and validation.

This audit supports the claim that the reported grouped validation result is based on the stated decision-time features rather than future information.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Leakage audit

final_features = [
    "days_since_last_update",
    "search_volume",
    "impressions_last_30d",
    "avg_position"
]

# Fields that would be suspicious if used as model inputs
future_or_outcome_fields = [
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "trend_pct",
    "trend_direction"
]

leaked_features = [
    col for col in final_features
    if col in future_or_outcome_fields
]

print("=== LEAKAGE AUDIT ===")
print("Final model features:")
for feature in final_features:
    print(" -", feature)

print("\nSuspicious future/outcome fields used:", leaked_features)

if len(leaked_features) == 0:
    print("\n✅ Leakage check passed.")
    print("No listed future/outcome fields are used as model inputs.")
else:
    print("\n❌ Potential leakage detected.")

=== LEAKAGE AUDIT ===
Final model features:
 - days_since_last_update
 - search_volume
 - impressions_last_30d
 - avg_position

Suspicious future/outcome fields used: []

✅ Leakage check passed.
No listed future/outcome fields are used as model inputs.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
### Claim rewrite

My original claim was that the Random Forest could improve the baseline action score.

A safer claim is:

"On the available 30,000-row dataset, the Random Forest was evaluated using a grouped client split with no client overlap. The measured MAE was 0.006802 and RMSE was 0.016359, compared with 0.000000 for the Week-4 baseline when predicting the baseline score itself. In this comparison, the Random Forest did not outperform the baseline. The result is directional decision-support evidence rather than proof of production performance or future content-refresh outcomes."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Claim rewrite check

print("=== CLAIM REWRITE ===")
print(
    "The Random Forest did not outperform the Week-4 baseline "
    "in the grouped validation comparison."
)
print(
    "This is directional decision-support evidence, "
    "not a production-performance or causal claim."
)

print("\n=== KEY VALIDATION FACTS ===")
print("Validation rows:", len(X_test))
print("Client overlap:", len(train_clients.intersection(test_clients)))
print("Baseline MAE:", f"{baseline_mae:.6f}")
print("Random Forest MAE:", f"{model_mae:.6f}")
print("Baseline RMSE:", f"{baseline_rmse:.6f}")
print("Random Forest RMSE:", f"{model_rmse:.6f}")

=== CLAIM REWRITE ===
The Random Forest did not outperform the Week-4 baseline in the grouped validation comparison.
This is directional decision-support evidence, not a production-performance or causal claim.

=== KEY VALIDATION FACTS ===
Validation rows: 6163
Client overlap: 0
Baseline MAE: 0.000000
Random Forest MAE: 0.006802
Baseline RMSE: 0.000000
Random Forest RMSE: 0.016359


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.